# Lesson 5A: K-Nearest Neighbors Theory

## Table of Contents1. Introduction2. Distance Metrics3. Curse of Dimensionality4. KD-Tree5. Bias-Variance6. Implementations7. Applications

## IntroductionK-Nearest Neighbors (KNN) is a simple yet powerful lazy learning algorithm. The core idea: to predict a new point's label, examine its k nearest neighbors in training data and use majority vote.Key characteristics:- Instance-based learning- No parametric assumptions- Non-parametric- Locally adaptive- Interpretable

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_iris, make_classificationfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitimport timeimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette('husl')np.random.seed(42)print('Libraries imported!')

## Distance MetricsThe foundation of KNN lies in distance calculation.

In [ ]:
# Distance metric implementationsdef euclidean_distance(x1, x2):    return np.sqrt(np.sum((x1 - x2) ** 2))def manhattan_distance(x1, x2):    return np.sum(np.abs(x1 - x2))def chebyshev_distance(x1, x2):    return np.max(np.abs(x1 - x2))def minkowski_distance(x1, x2, p=2):    return np.sum(np.abs(x1 - x2) ** p) ** (1.0 / p)# Example calculationsx1, x2 = np.array([0, 0]), np.array([3, 4])print('Distance Examples:')print(f'Euclidean: {euclidean_distance(x1, x2):.4f}')print(f'Manhattan: {manhattan_distance(x1, x2):.4f}')print(f'Chebyshev: {chebyshev_distance(x1, x2):.4f}')print(f'Minkowski(p=3): {minkowski_distance(x1, x2, p=3):.4f}')

# Extended Euclidean variants for special cases
def euclidean_distance_weighted(x1, x2, weights=None):
    """Weighted Euclidean distance."""
    if weights is None:
        weights = np.ones_like(x1)
    return np.sqrt(np.sum(weights * (x1 - x2) ** 2))

def euclidean_distance_normalized(x1, x2):
    """Normalized Euclidean distance (unit vectors)."""
    n1 = x1 / (np.linalg.norm(x1) + 1e-10)
    n2 = x2 / (np.linalg.norm(x2) + 1e-10)
    return euclidean_distance(n1, n2)

# Validation: Extended distance functions
x1, x2 = np.array([1, 2, 3]), np.array([4, 5, 6])
weights = np.array([1, 2, 1])

print("Extended Euclidean distance variants:")
print(f"  Standard: {euclidean_distance(x1, x2):.4f}")
print(f"  Weighted: {euclidean_distance_weighted(x1, x2, weights):.4f}")
print(f"  Normalized: {euclidean_distance_normalized(x1, x2):.4f}")


In [ ]:
# Vectorized distance computation (essential for efficiency)def euclidean_vectorized(X, y):    X_sqnorm = np.sum(X ** 2, axis=1, keepdims=True)    y_sqnorm = np.sum(y ** 2)    X_dot_y = np.dot(X, y)    distances_sq = X_sqnorm.ravel() + y_sqnorm - 2 * X_dot_y    return np.sqrt(np.maximum(distances_sq, 0))# BenchmarkX = np.random.randn(10000, 100)y = np.random.randn(100)start = time.time()for _ in range(10):    _ = euclidean_vectorized(X, y)vec_time = time.time() - startprint(f'Vectorized 10 queries on 10000 points: {vec_time:.4f}s')

In [ ]:
# Visualize different distance metricsfig, axes = plt.subplots(1, 3, figsize=(15, 4))theta = np.linspace(0, 2*np.pi, 100)# Euclideanaxes[0].plot(5*np.cos(theta), 5*np.sin(theta), 'b-', linewidth=2)axes[0].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[0].set_title('Euclidean (Circle)')axes[0].set_aspect('equal')axes[0].grid(True, alpha=0.3)# Manhattanmanhattan_pts = [[5, 0], [0, 5], [-5, 0], [0, -5], [5, 0]]manhattan_pts = np.array(manhattan_pts)axes[1].plot(manhattan_pts[:, 0], manhattan_pts[:, 1], 'r-', linewidth=2)axes[1].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[1].set_title('Manhattan (Diamond)')axes[1].set_aspect('equal')axes[1].grid(True, alpha=0.3)# Chebyshevchebyshev_pts = [[5, 5], [-5, 5], [-5, -5], [5, -5], [5, 5]]chebyshev_pts = np.array(chebyshev_pts)axes[2].plot(chebyshev_pts[:, 0], chebyshev_pts[:, 1], 'g-', linewidth=2)axes[2].scatter([0], [0], c='red', s=100, marker='x', zorder=5)axes[2].set_title('Chebyshev (Square)')axes[2].set_aspect('equal')axes[2].grid(True, alpha=0.3)for ax in axes:    ax.set_xlim(-6, 6)    ax.set_ylim(-6, 6)plt.suptitle('Distance Metrics: Points at distance 5 from origin', fontweight='bold')plt.tight_layout()plt.show()

## Curse of DimensionalityAs dimensions increase, all distances become approximately equal.

In [ ]:
# Distance concentration phenomenondef distance_concentration(n_samples=1000, n_trials=3):    dimensions = np.arange(1, 51, 10)    ratios = []    for d in dimensions:        d_ratios = []        for _ in range(n_trials):            X = np.random.uniform(0, 1, (n_samples, d))            query = np.random.uniform(0, 1, d)            dist = euclidean_vectorized(X, query)            ratio = np.max(dist) / (np.min(dist) + 1e-10)            d_ratios.append(ratio)        ratios.append(np.mean(d_ratios))    return dimensions, ratiosprint('Computing distance concentration...')dims, ratios = distance_concentration()print('\nDimension-Distance Ratio:')for d, r in zip(dims, ratios):    print(f'  d={d:2d}: ratio={r:8.2f}')# Visualizefig, ax = plt.subplots(figsize=(10, 6))ax.semilogy(dims, ratios, 'o-', linewidth=2, markersize=8, color='darkred')ax.axhline(y=1, color='green', linestyle='--', linewidth=2, label='All distances equal')ax.set_xlabel('Dimensions')ax.set_ylabel('Max Distance / Min Distance (log)')ax.set_title('Distance Concentration: All Distances Become Equal in High Dimensions')ax.grid(True, alpha=0.3, which='both')ax.legend()plt.tight_layout()plt.show()

## KD-Tree: Efficient Nearest Neighbor SearchKD-trees enable O(log n) search instead of O(n) brute force.

In [ ]:
# KD-Tree Nodeclass KDTreeNode:    def __init__(self, point=None, axis=None, left=None, right=None):        self.point = point        self.axis = axis        self.left = left        self.right = right# KD-Tree classclass KDTree:    def __init__(self, X):        self.X = X        self.root = self._build(np.arange(len(X)), axis=0)        self.n_features = X.shape[1]    def _build(self, indices, axis=0):        if len(indices) == 0:            return None        axis_vals = self.X[indices, axis % self.n_features]        median_pos = np.argsort(axis_vals)[len(indices) // 2]        median_idx = indices[median_pos]        axis_val = self.X[median_idx, axis % self.n_features]        left_idx = indices[self.X[indices, axis % self.n_features] < axis_val]        right_idx = indices[self.X[indices, axis % self.n_features] >= axis_val]        return KDTreeNode(            point=self.X[median_idx],            axis=axis % self.n_features,            left=self._build(left_idx, axis + 1),            right=self._build(right_idx, axis + 1)        )    def query(self, point, k=1):        best = []        self._search(self.root, point, k, best)        return sorted(best, key=lambda x: x[1])[:k]    def _search(self, node, point, k, best):        if node is None:            return        dist = euclidean_distance(point, node.point)        if len(best) < k:            best.append((node.point.copy(), dist))            best.sort(key=lambda x: x[1])        elif dist < best[-1][1]:            best[-1] = (node.point.copy(), dist)            best.sort(key=lambda x: x[1])        axis = node.axis        if point[axis] < node.point[axis]:            near, far = node.left, node.right        else:            near, far = node.right, node.left        self._search(near, point, k, best)        if len(best) < k or abs(point[axis] - node.point[axis]) < best[-1][1]:            self._search(far, point, k, best)# Testnp.random.seed(42)X_kd = np.random.randn(100, 2) * 10tree = KDTree(X_kd)query = np.array([0.0, 0.0])neighbors = tree.query(query, k=5)print(f'Query: {query}')print(f'5 nearest neighbors:')for i, (p, d) in enumerate(neighbors, 1):    print(f'  {i}. {p}: distance={d:.4f}')

# Extended Euclidean variants for special cases
def euclidean_distance_weighted(x1, x2, weights=None):
    """Weighted Euclidean distance."""
    if weights is None:
        weights = np.ones_like(x1)
    return np.sqrt(np.sum(weights * (x1 - x2) ** 2))

def euclidean_distance_normalized(x1, x2):
    """Normalized Euclidean distance (unit vectors)."""
    n1 = x1 / (np.linalg.norm(x1) + 1e-10)
    n2 = x2 / (np.linalg.norm(x2) + 1e-10)
    return euclidean_distance(n1, n2)

# Validation: Extended distance functions
x1, x2 = np.array([1, 2, 3]), np.array([4, 5, 6])
weights = np.array([1, 2, 1])

print("Extended Euclidean distance variants:")
print(f"  Standard: {euclidean_distance(x1, x2):.4f}")
print(f"  Weighted: {euclidean_distance_weighted(x1, x2, weights):.4f}")
print(f"  Normalized: {euclidean_distance_normalized(x1, x2):.4f}")


## KNN Classifier Implementation

In [ ]:
class KNNClassifier:    def __init__(self, k=5, metric='euclidean'):        self.k = k        self.metric = metric        self.X_train = None        self.y_train = None    def fit(self, X, y):        self.X_train = X.astype(np.float64)        self.y_train = y        return self    def _distances(self, X_test):        if self.metric == 'euclidean':            return euclidean_vectorized(self.X_train, X_test)        else:            return np.sum(np.abs(self.X_train - X_test), axis=1)    def predict(self, X_test):        predictions = []        for x in X_test:            dist = self._distances(x)            k_idx = np.argsort(dist)[:self.k]            labels = self.y_train[k_idx]            unique, counts = np.unique(labels, return_counts=True)            predictions.append(unique[np.argmax(counts)])        return np.array(predictions)    def score(self, X_test, y_test):        return np.mean(self.predict(X_test) == y_test)# Test on Irisiris = load_iris()X, y = iris.data, iris.targetX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train = scaler.fit_transform(X_train)X_test = scaler.transform(X_test)knn = KNNClassifier(k=5)knn.fit(X_train, y_train)print(f'KNN Classifier Test (Iris):')print(f'Train accuracy: {knn.score(X_train, y_train):.4f}')print(f'Test accuracy: {knn.score(X_test, y_test):.4f}')

## Bias-Variance TradeoffSmall k: low bias, high variance (overfitting)Large k: high bias, low variance (underfitting)

In [ ]:
# Test different k valuesk_values = [1, 3, 5, 7, 10, 15]results = []for k in k_values:    knn_k = KNNClassifier(k=k)    knn_k.fit(X_train, y_train)    train_acc = knn_k.score(X_train, y_train)    test_acc = knn_k.score(X_test, y_test)    results.append((k, train_acc, test_acc))print('k  | Train Acc | Test Acc')print('-' * 28)for k, train, test in results:    print(f'{k:2d} | {train:9.4f} | {test:8.4f}')# Visualizeks = [r[0] for r in results]trains = [r[1] for r in results]tests = [r[2] for r in results]fig, ax = plt.subplots(figsize=(10, 6))ax.plot(ks, trains, 'o-', linewidth=2, markersize=8, label='Training')ax.plot(ks, tests, 's-', linewidth=2, markersize=8, label='Test')ax.set_xlabel('k (number of neighbors)')ax.set_ylabel('Accuracy')ax.set_title('KNN Performance vs k')ax.legend()ax.grid(True, alpha=0.3)ax.set_xticks(ks)plt.tight_layout()plt.show()

## Complexity AnalysisBrute force: O(1) training, O(n) queryKD-tree: O(n log n) training, O(log n) query

## Extended Background: Why KNN MattersK-Nearest Neighbors represents a fundamentally different paradigm from parametric models. Where Support Vector Machines or Neural Networks learn explicit decision boundaries during training, KNN stores the training data and makes predictions by examining local neighborhoods.This lazy learning approach has profound implications:### Memory-Based LearningKNN is an instance-based or memory-based learning algorithm. The entire training dataset becomes the model—prediction time computation replaces training time computation. This trades off fast training for slower prediction but gains important advantages:1. **Naturally incremental**: New training data can be added without retraining2. **Interpretable**: Predictions explained by examining actual training examples3. **Non-parametric**: No assumptions about data distribution shape4. **Adaptive**: Decision boundaries adapt locally to data density### Why Distance Metrics MatterThe entire algorithm hinges on "distance." Computing distances between vectors is not trivial—the choice of metric shapes everything:- Different metrics create different geometric notions of "closeness"- Features with different scales can dominate if not normalized- High-dimensional spaces create pathological distance behavior- The metric defines what "neighbors" means—fundamentally shaping predictions### The Curse of DimensionalityThis is KNN's Achilles heel. As dimensions increase:- All distances concentrate toward a constant value- The distinction between near and far neighbors disappears- More samples needed to maintain sparsity- Computational cost remains high despite mathematical ineffectivenessUnderstanding these factors is essential for using KNN effectively.

## Mathematical Foundations of DistanceA distance metric d(x,y) is a function satisfying four axioms:1. **Non-negativity**: d(x,y) ≥ 0 for all x,y2. **Identity of indiscernibles**: d(x,y) = 0 if and only if x = y3. **Symmetry**: d(x,y) = d(y,x) for all x,y4. **Triangle inequality**: d(x,z) ≤ d(x,y) + d(y,z) for all x,y,zAny function satisfying these axioms is called a **metric** and defines a valid geometric space with well-defined geometric properties.### Norm-Induced MetricsMost metrics used in machine learning are induced by norms. A norm ||·|| is a function satisfying:- ||x|| ≥ 0 (non-negative)- ||x|| = 0 ⟺ x = 0 (positive definite)- ||αx|| = |α| ||x|| (homogeneous)- ||x + y|| ≤ ||x|| + ||y|| (triangle inequality)The metric induced by norm ||·|| is:d(x,y) = ||x - y||All Lp norms induce metrics:||x||_p = (Σ_i |x_i|^p)^(1/p)

In [ ]:
# Mathematical properties verificationimport numpy as npdef verify_metric_properties(x1, x2, x3, d_func):    print("Verifying metric properties for distance function:")        # Non-negativity    d12 = d_func(x1, x2)    print(f"1. Non-negativity: d(x1,x2) = {d12:.4f} >= 0? {d12 >= 0}")        # Identity (using same point)    d11 = d_func(x1, x1)    print(f"2. Identity: d(x1,x1) = {d11:.6f} = 0? {abs(d11) < 1e-10}")        # Symmetry    d21 = d_func(x2, x1)    print(f"3. Symmetry: d(x1,x2) = {d12:.4f}, d(x2,x1) = {d21:.4f}? {abs(d12-d21) < 1e-10}")        # Triangle inequality    d13 = d_func(x1, x3)    d23 = d_func(x2, x3)    print(f"4. Triangle: d(x1,x3) = {d13:.4f} <= d(x1,x2) + d(x2,x3) = {d12:.4f} + {d23:.4f} = {d12+d23:.4f}? {d13 <= d12+d23+1e-10}")x1 = np.array([0, 0, 0])x2 = np.array([3, 4, 0])x3 = np.array([1, 1, 1])print("="*70)print("EUCLIDEAN METRIC PROPERTIES")print("="*70)verify_metric_properties(x1, x2, x3, euclidean_distance)print("\n" + "="*70)print("MANHATTAN METRIC PROPERTIES")print("="*70)verify_metric_properties(x1, x2, x3, manhattan_distance)

# Extended Euclidean variants for special cases
def euclidean_distance_weighted(x1, x2, weights=None):
    """Weighted Euclidean distance."""
    if weights is None:
        weights = np.ones_like(x1)
    return np.sqrt(np.sum(weights * (x1 - x2) ** 2))

def euclidean_distance_normalized(x1, x2):
    """Normalized Euclidean distance (unit vectors)."""
    n1 = x1 / (np.linalg.norm(x1) + 1e-10)
    n2 = x2 / (np.linalg.norm(x2) + 1e-10)
    return euclidean_distance(n1, n2)

# Validation: Extended distance functions
x1, x2 = np.array([1, 2, 3]), np.array([4, 5, 6])
weights = np.array([1, 2, 1])

print("Extended Euclidean distance variants:")
print(f"  Standard: {euclidean_distance(x1, x2):.4f}")
print(f"  Weighted: {euclidean_distance_weighted(x1, x2, weights):.4f}")
print(f"  Normalized: {euclidean_distance_normalized(x1, x2):.4f}")


## KNN Decision Boundary AnalysisFor a k-NN classifier, the decision boundary in 2D can be visualized and analyzed mathematically.### Voronoi Diagrams and k-NNEach training point has an associated Voronoi cell—the region closer to that point than any other. With k=1, the decision boundary coincides with the Voronoi diagram edges.For larger k, the decision boundary becomes smoother as votes are aggregated from multiple neighbors.### Asymptotic BehaviorThe classification error of k-NN approaches the Bayes error rate as:- n → ∞ (number of training samples)- k → ∞- k/n → 0 (k grows but slower than n)This makes k-NN a **consistent** classifier—given enough data, it approaches optimal performance.

In [ ]:
# Summaryprint('=' * 70)print('KNN SUMMARY')print('=' * 70)print('''ADVANTAGES:- Simple and intuitive- Non-parametric- No training phase- InterpretableDISADVANTAGES:- Slow prediction (O(n) brute force)- Sensitive to distance metric- Curse of dimensionality- Stores all training dataWHEN TO USE:- Small to medium datasets- Low-moderate dimensions- Non-linear boundaries needed- Interpretability importantWHEN TO AVOID:- Very large datasets- High dimensions- Real-time required- Need probabilistic outputs''')

print(f"\n{'='*70}")
print("COMPREHENSIVE KNN SUMMARY")
print(f"{'='*70}")

summary = """
KNN ALGORITHM PSEUDOCODE:
========================

procedure KNearestNeighbors(X_train, y_train, query, k):
    for each training sample x_i in X_train:
        distance[i] = distance_metric(query, x_i)
    
    k_min_indices = k indices with minimum distances
    k_min_labels = y_train[k_min_indices]
    
    prediction = majority_vote(k_min_labels)
    return prediction

TIME COMPLEXITY:
- Training: O(1) - just store data
- Single query: O(n*d) with brute force
- k queries: O(n*d*k)
- With KD-tree: O(log n) average, O(n) worst case

SPACE COMPLEXITY:
- O(n*d) to store training data

PARAMETERS:
- k: number of neighbors (critical for bias-variance)
- distance metric: determines geometric space
- weights: uniform vs distance-weighted voting
- normalization: feature scaling essential

BEST PRACTICES:
1. Normalize/standardize all features
2. Use cross-validation to select k
3. Consider distance weighting
4. Use KD-trees for n > 1000
5. Reduce dimensions if d > 50
6. Handle class imbalance with weighted voting
"""

print(summary)

# Guidelines for parameter selection
print("\nPARAMETER SELECTION GUIDELINES:")
print("="*70)
print("k selection:")
print("  - Rule of thumb: k = sqrt(n)")
print("  - For classification: k=3,5,7 often good")
print("  - For regression: k=n/10")
print("  - Always use cross-validation!")
print("\nDistance metric:")
print("  - Euclidean: default, works for continuous features")
print("  - Manhattan: robust to outliers, discrete features")
print("  - Minkowski: general parametric approach")
print("  - Mahalanobis: correlated features")
print("\nFeature scaling:")
print("  - MANDATORY: standardize all features")
print("  - Use StandardScaler or similar")
print("  - Different scales distort distance computations")


## Practical Applications of K-NN### Real-World Use Cases#### 1. Image Classification- Find similar images in database- Face recognition and verification- Object detection in images#### 2. Recommendation Systems- Collaborative filtering (users similar to you liked X)- Content-based filtering- Hybrid approaches#### 3. Medical Diagnosis- Diagnosis based on patient similarity- Disease prognosis from similar cases- Treatment recommendation#### 4. Time Series Analysis- Anomaly detection (unusual patterns)- Forecasting based on historical similarity- Pattern recognition#### 5. Natural Language Processing- Document classification- Sentiment analysis- Machine translation (example-based)### Industry Examples**Netflix**: Recommendation engine uses k-NN variants to suggest movies to users**Amazon**: Product recommendations based on similar user behavior**Google**: Similar page discovery uses k-NN concepts**Medical Diagnosis**: IBM Watson uses k-NN for disease matching**Financial**: Fraud detection by finding anomalous transactions similar to known fraud

## ConclusionKNN is a powerful, interpretable algorithm. Key considerations:- Choose k via cross-validation- Normalize features- Use KD-trees for speedup- Beware curse of dimensionality

## Comprehensive Summary

K-Nearest Neighbors is a fundamental distance-based learning algorithm. Key concepts covered:

### Distance Metrics
- **Lp Norms**: Family of distance functions with mathematical properties
- **Euclidean**: Most common, geometric interpretation, O(1) to compute
- **Manhattan**: Robust alternative, taxicab geometry
- **Mahalanobis**: Accounts for correlations, optimal for Gaussian data

### Curse of Dimensionality
- Volume concentration: Most volume in corners of hypercube in high d
- Distance concentration: All points become nearly equidistant
- Sample complexity: Need n ~ (1/ε)^d for fixed density
- Practical impact: KNN fails for d > 50 without dimensionality reduction

### KD-Tree Optimization
- Binary space partitioning along each axis
- Construction: O(n log n), Query: O(log n) average
- Pruning strategy: Skip branches far from query point
- When to use: d < 20 and uniformly distributed data

### Bias-Variance Tradeoff
- Small k: Low bias, high variance (overfitting)
- Large k: High bias, low variance (underfitting)
- Optimal k: Found via cross-validation
- Heuristic: k ≈ √n for classification

### Practical Considerations
- Feature normalization: ESSENTIAL for fair distance computation
- Feature selection: Critical for high-dimensional data
- Weighted voting: Improves performance by giving closer points more influence
- Cross-validation: Required for hyperparameter tuning

### When to Use KNN
- ✓ Small to medium datasets (n < 100k)
- ✓ Low dimensions (d < 20)
- ✓ Local patterns important
- ✓ Interpretability needed
- ✗ High dimensions
- ✗ Real-time predictions
- ✗ Massive datasets

## Mathematical Properties of Distance Functions

### Metric Axioms
A valid distance function d(x,y) must satisfy:
1. d(x,y) ≥ 0 (non-negative)
2. d(x,y) = 0 ⟺ x = y (identity)
3. d(x,y) = d(y,x) (symmetry)
4. d(x,z) ≤ d(x,y) + d(y,z) (triangle inequality)

These axioms ensure that KNN produces well-defined neighborhoods.

### Norm-Induced Metrics
Metrics derived from norms satisfy all metric axioms automatically:
- Norm definition: ‖x‖ satisfies non-negativity, homogeneity, triangle inequality
- Induced metric: d(x,y) = ‖x - y‖
- Result: Valid metric for KNN use

### Lp Norm Mathematical Properties
The Lp norm: ‖x‖_p = (∑ᵢ |xᵢ|^p)^(1/p)

- Monotonic in p: ‖x‖_1 ≥ ‖x‖_2 ≥ ‖x‖_∞
- Continuous in p: ‖x‖_p changes smoothly with p
- Limits: lim_{p→∞} ‖x‖_p = max_i |xᵢ|
- Relationship: All valid metrics for distance computation

## Curse of Dimensionality: Detailed Analysis

### Volume Concentration
In a d-dimensional unit cube [0,1]^d:
- Volume of cube: 1
- Volume of inscribed sphere (radius 0.5): π^(d/2) / (2^d * Γ(d/2+1))
- Ratio: → 0 exponentially as d → ∞

**Consequence**: Most of the cube's volume is in the corners, not near the center.

### Distance Concentration for Random Points
For n random points uniformly distributed in [0,1]^d:
- Nearest neighbor distance: ~ (1/2)^(1/d)
- Farthest neighbor distance: ~ √d
- Ratio: ~ 2^(d/2) * √d → ∞ exponentially

### Impact on KNN
1. All k nearest neighbors nearly equally distant
2. Neighborhood concept loses meaning
3. Votes are from essentially random points
4. Predictions become unreliable
5. Need exponentially more training data

### Solutions
- Feature selection: Keep only relevant features
- Dimensionality reduction: PCA, t-SNE, manifold learning
- Alternative algorithms: For d > 50, use methods more suitable to high dimensions
- Approximate methods: LSH, ball-trees for scalability

## Implementation Recommendations

### Feature Preprocessing
**Normalization** (make features [0,1]):
```
x_normalized = (x - x_min) / (x_max - x_min)
```

**Standardization** (zero mean, unit variance):
```
x_standardized = (x - mean) / std
```

**Why essential**: Without preprocessing, features with larger scales dominate distance computation.

### Hyperparameter Selection
**Grid Search with Cross-Validation**:
1. Candidate k values: {1, 3, 5, 7, 9, 11, 15}
2. For each k: 5-fold cross-validation
3. Select k with lowest CV error
4. Evaluate on held-out test set

### Distance Metric Selection
- **Default**: Euclidean for continuous features
- **Outliers**: Manhattan for robustness
- **Correlated features**: Mahalanobis
- **Text/Sparse**: Cosine similarity
- **Binary**: Hamming distance

### Optimization Techniques
- **KD-trees**: For d < 20, n > 1000
- **Ball-trees**: More robust for varying dimensions
- **Approximate NN**: For very large n (n > 1M)
- **Weighted voting**: Inverse distance or Gaussian kernel

## Theoretical Guarantees

### Consistency
k-NN is a **consistent** classifier: As n → ∞ with k → ∞ and k/n → 0,
the misclassification error approaches the Bayes error rate (optimal possible error).

**Practical implication**: Given enough data, k-NN can be as good as any classifier.

### Asymptotic Behavior
For large n and optimal k:
- Training error: 0 (if k ≤ n)
- Test error: Approaches Bayes error
- Convergence rate: Depends on dimensionality

### Finite Sample Analysis
With finite n and d-dimensional features:
- Sample complexity: O(1/ε)^(d/c) where c depends on problem structure
- Curse of dimensionality: Exponential dependence on d
- Implies: Dimensionality reduction often necessary in practice

### Decision Boundary Properties
For k-NN classifier with k=1:
- Decision boundaries are piecewise linear
- Defined by perpendicular bisectors of training point pairs
- Can approximate any decision boundary with enough training data
- For k>1: Boundaries become smoother (more generalization)

## Comparison with Other Algorithms

### vs. Parametric Methods (Logistic Regression, SVM)
- KNN: Non-parametric, no training phase
- Parametric: Learn fixed decision boundary during training
- Trade-off: KNN slow prediction but adaptive; parametric fast prediction

### vs. Tree-Based Methods (Decision Trees, Random Forests)
- KNN: Local averaging, smooth predictions
- Trees: Hierarchical partitioning, piecewise constant
- KNN better for: Smooth boundaries
- Trees better for: Categorical features, interpretability

### vs. Neural Networks
- KNN: Simple, interpretable, no hyperparameters beyond k
- NN: Complex, learns hierarchical features
- KNN better for: Small datasets, interpretability
- NN better for: Large datasets, complex patterns

### When to Choose KNN
**Choose KNN when:**
- Dataset is small to medium (n < 100k)
- Need interpretable predictions
- Don't know the structure of decision boundary
- Computational cost of training less important than inference cost

**Avoid KNN when:**
- Dataset is very large (n > 1M)
- Real-time prediction required
- Dimensionality is very high (d > 50)
- Memory is limited (KNN stores all training data)

## Conclusion

### Key Takeaways
1. **Simple but powerful**: KNN is non-parametric and requires no training
2. **Distance matters**: Choice of metric fundamentally affects performance
3. **Curse of dimensionality**: KNN degrades badly in high dimensions
4. **k selection critical**: Use cross-validation, never guess
5. **Preprocessing essential**: Always normalize/standardize features
6. **Trade-offs**: Simple algorithm with clear advantages and limitations

### When to Use
- ✓ Small datasets with clear local patterns
- ✓ Need interpretable predictions
- ✓ Features are continuous and well-scaled
- ✗ Large datasets or high dimensions
- ✗ Real-time predictions required
- ✗ Sparse or categorical data

### Further Study
- **Advanced metrics**: Learn about domain-specific distances
- **Dimensionality reduction**: PCA, manifold learning techniques
- **Approximate NN**: Locality-sensitive hashing for massive datasets
- **Kernel methods**: Extensions of KNN concepts
- **Ensemble methods**: Combining KNN with other algorithms

This concludes the comprehensive K-Nearest Neighbors theory notebook.

## Weighted KNN Implementation Details
A complete weighted k-NN implementation must handle:
1. Multiple distance metrics (Euclidean, Manhattan, etc.)
2. Different weighting schemes (uniform, inverse, Gaussian)
3. Efficient distance computation for large datasets
4. Proper handling of ties in voting
5. Support for both classification and regression
Key implementation considerations:
- Compute all distances once, then extract k best
- Normalize weights to handle zero-distance cases
- Use argsort for efficient k selection
- Cache distance computations when possible
## Performance Analysis and Optimization
### Time Complexity Analysis
Brute Force Nearest Neighbor Search:
- Training time: O(1) - just store training data
- Single query: O(n * d) where n is samples, d is features
- k queries: O(n * d * k)
- Sorting k elements: O(n log k)
KD-Tree Search:
- Construction: O(n log n) via median partitioning
- Average query: O(log n + k) - go to k nearest then backtrack
- Worst case: O(n) - when must search entire tree
- Effective when: d < 20 and data uniformly distributed


- Degrades when: d > 50 or data very skewed
Ball-Tree Search:
- Construction: O(n log n)
- Query: O(log n) average, O(n) worst case
- More robust to high dimensions than KD-tree
- Better for: 20 < d < 100
### When Each Method is Optimal
Use brute force when:
- Dataset small (n < 1000)
- Query count small (< 10)
- Memory is critical
Use KD-tree when:
- Low dimensions (d < 20)
- Large dataset (n > 1000)
- Uniformly distributed data
- Many queries on same data
Use Ball-tree when:
- Moderate to high dimensions (20 < d < 100)
- Non-uniform or clustered data
- Want robust performance
Use approximate methods when:
- Very large dataset (n > 1M)
- High dimensions (d > 100)
- Real-time requirements


- Can tolerate approximate results
## Cross-Validation Best Practices
### Proper Cross-Validation Setup
1. Normalize BEFORE splitting into folds
2. Never normalize using statistics from test set
3. Use stratified k-fold for imbalanced datasets
4. Use same preprocessing in each fold
### Grid Search Procedure
1. Define candidate hyperparameters
2. For each candidate:
   a. Perform k-fold cross-validation
   b. Record mean and standard deviation
   c. Note validation error curve behavior
3. Select hyperparameter with lowest CV error
4. Retrain on full training set with selected hyperparameter
5. Report final performance on held-out test set
### Avoiding Common Pitfalls
- Don't use test set to select k (data leakage)
- Don't normalize using test set statistics
- Don't select k with highest training accuracy (overfitting)
- Do use cross-validation for honest estimates
- Do keep validation and test sets separate


## Complete KNN Algorithm Pseudocode
```
ALGORITHM: K-Nearest Neighbors Classification
INPUT: Training set (X_train, y_train), query x, parameter k, distance metric d
OUTPUT: Predicted class label for x
1. Compute distances from x to all training samples:
   distances[i] = d(x, X_train[i]) for i = 1 to n
2. Find k indices with smallest distances:
   k_indices = indices of k smallest distances
3. Get labels of k nearest neighbors:
   k_labels = y_train[k_indices]
4. Count votes from k neighbors:
   for each unique label c:
       votes[c] = count of c in k_labels
5. Return label with maximum votes:
   return argmax(votes)
```
## Practical Implementation Tips
### Feature Preprocessing is Essential
Without proper scaling, features with larger ranges dominate distances:


Example: House features
- Square footage: typically 1000-5000
- Number of bathrooms: typically 1-5
- Distance dominated by square footage despite both being important
Solution: Standardization or normalization
- StandardScaler: (x - mean) / std
- MinMaxScaler: (x - min) / (max - min)
### Handling Categorical Features
KNN with distance metrics requires numerical features.
For categorical data:
- One-hot encoding: Convert to binary vectors
- Ordinal encoding: If ordering exists
- Use distance metrics designed for mixed data
### Dealing with Missing Values
Options for missing data:
1. Remove samples with missing values (data loss)
2. Impute missing values (mean, median, KNN imputation)
3. Use algorithms robust to missingness
KNN imputation: Use KNN to estimate missing values from similar samples
### Memory Considerations
KNN stores entire training set in memory.


For large datasets:
- Consider dimensionality reduction
- Use approximate nearest neighbor methods
- Implement mini-batch versions
- Consider more memory-efficient algorithms
## Conclusion and Summary
K-Nearest Neighbors is a powerful yet simple algorithm with important limitations:
Advantages:
- Non-parametric: learns flexible decision boundaries
- No training phase: instant adaptation to new data
- Interpretable: explanations from nearest neighbors
- Consistent: approaches Bayes error with enough data
Disadvantages:
- Slow prediction: O(n) brute force or O(log n) with trees
- Memory intensive: must store all training data
- Curse of dimensionality: fails in high dimensions
- Sensitive to feature scaling: requires careful preprocessing
Best used for:
- Small to medium datasets (n < 100,000)
- Low to moderate dimensions (d < 20)
- Non-linear decision boundaries
- When interpretability is critical
Avoid for:
- Very large datasets


- High-dimensional data
- Real-time prediction requirements
- Sparse or categorical data without preprocessing
